<a href="https://colab.research.google.com/github/zahere-dev/advanced-rag/blob/main/01-chunking/fixed_size_chunking.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Fixed-Size Chunking

Part of the RAG Master Class: https://www.youtube.com/watch?v=hEG3-L4l5R4

## Install required packages

In [ ]:
!pip install langchain langchain_community langchain-experimental langchain-openai chromadb datasets ragas

In [3]:
import bs4
from langchain.text_splitter import CharacterTextSplitter
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain.document_loaders import TextLoader

## Load the dataset

In [ ]:
loader = TextLoader("/content/medical_case_studies_synthetica_dataset.txt")
documents = loader.load()
documents

In [ ]:
chunk_size = 1000
chunk_overlap = 200

text_splitter = CharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap, separator='\n', strip_whitespace=False)

# Split each sentence/paragraph into chunks
fixed_size_chunks = text_splitter.create_documents([d.page_content for d in documents])


for i in range(10):
  print("Start_of_new_chunk"+ "*-*-"*50)
  print(fixed_size_chunks[i])
  print("End_of_chunk"+ "*-*-"*50)
  print("\n")

In [15]:
import os
from langchain_openai.embeddings import OpenAIEmbeddings
from google.colab import userdata
from langchain_community.vectorstores import Chroma

OPENAI_API_KEY = os.environ["OPENAI_API_KEY"] =  userdata.get("OPENAI_API_KEY")

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
fixed_sized_chunk_vectorstore = Chroma.from_documents(fixed_size_chunks, embedding=embeddings)

In [16]:
fixed_sized_chunk_retriever = fixed_sized_chunk_vectorstore.as_retriever(search_kwargs={"k" : 1})
fixed_sized_chunk_retriever.invoke("How are nanoparticles being used in Alzheimer's disease treatment??")

[Document(metadata={}, page_content="      - 38% decrease in pro-inflammatory cytokines\n   d) Safety:\n      - No significant toxicity observed\n      - Normal liver and kidney function tests\n3. Comparison to Control Groups:\n   - Significantly superior Aβ clearance and cognitive improvements compared to all control groups (p<0.001)\n   - Empty nanoparticles and free neprilysin showed minimal effects\nDiscussion:\nThis study demonstrates the potential of nanotechnology-based approaches for the treatment of Alzheimer's Disease. The designed nanoparticles successfully crossed the blood-brain barrier and delivered functional Aβ-degrading enzyme to the brain. The significant reduction in Aβ levels, coupled with improved cognitive function and reduced neuroinflammation, suggests a promising therapeutic strategy.")]

In [17]:
from langchain_core.prompts import ChatPromptTemplate

rag_template = """\
Use the following context to answer the user's query. If you cannot answer, please respond with 'I don't know'.

User's Query:
{question}

Context:
{context}
"""

rag_prompt = ChatPromptTemplate.from_template(rag_template)

In [18]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini")

fixed_sized_rag_chain = (
    {"context" : fixed_sized_chunk_retriever, "question" : RunnablePassthrough()}
    | rag_prompt
    | llm
    | StrOutputParser()
)

In [19]:
synthetic_data_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=0,
    length_function=len,
    is_separator_regex=False
)
#
synthetic_data_chunks = synthetic_data_splitter.create_documents([d.page_content for d in documents])
print(len(synthetic_data_chunks))

69


In [20]:
questions = []
ground_truths_fixed_size = []
contexts = []
answers = []

question_prompt = """\
You are a teacher preparing a test. Please create a question that can be answered by referencing the following context.

Context:
{context}
"""

question_prompt = ChatPromptTemplate.from_template(question_prompt)

ground_truth_prompt = """\
Use the following context and question to answer this question using *only* the provided context.

Question:
{question}

Context:
{context}
"""

ground_truth_prompt = ChatPromptTemplate.from_template(ground_truth_prompt)

question_chain = question_prompt | llm | StrOutputParser()
ground_truth_chain = ground_truth_prompt | llm | StrOutputParser()

for chunk in synthetic_data_chunks[10:20]:
  questions.append(question_chain.invoke({"context" : chunk.page_content}))
  contexts.append([chunk.page_content])
  ground_truths_fixed_size.append(ground_truth_chain.invoke({"question" : questions[-1], "context" : contexts[-1]}))
  answers.append(fixed_sized_rag_chain.invoke(questions[-1]))

In [21]:
from datasets import load_dataset, Dataset

qagc_list = []

for question, answer, context, ground_truth in zip(questions, answers, contexts, ground_truths_fixed_size):
  qagc_list.append({
      "question" : question,
      "answer" : answer,
      "contexts" : context,
      "ground_truth" : ground_truth
  })

eval_dataset = Dataset.from_list(qagc_list)
eval_dataset

Dataset({
    features: ['question', 'answer', 'contexts', 'ground_truth'],
    num_rows: 10
})

In [22]:
from ragas.metrics import (
    answer_relevancy,
    faithfulness,
    context_recall,
    context_precision,
)

from ragas import evaluate

result = evaluate(
    eval_dataset,
    metrics=[
        context_precision,
        faithfulness,
        answer_relevancy,
        context_recall,
    ],
)
result

/usr/local/lib/python3.10/dist-packages/pydantic/_internal/_fields.py:132: UserWarning: Field "model_name" in _VertexAIBase has conflict with protected namespace "model_".

You may be able to resolve this warning by setting `model_config['protected_namespaces'] = ()`.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/pydantic/_internal/_fields.py:132: UserWarning: Field "model_name" in _VertexAICommon has conflict with protected namespace "model_".

You may be able to resolve this warning by setting `model_config['protected_namespaces'] = ()`.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/ragas/metrics/__init__.py:4: LangChainDeprecationWarning: As of langchain-core 0.3.0, LangChain uses pydantic v2 internally. The langchain_core.pydantic_v1 module was a compatibility shim for pydantic v1, and should no longer be used. Please update the code to import from Pydantic directly.

For example, replace imports like: `from langchain_core.pydantic_v1 import BaseModel`
with: `

Evaluating:   0%|          | 0/40 [00:00<?, ?it/s]

{'context_precision': 1.0000, 'faithfulness': 0.7385, 'answer_relevancy': 0.9183, 'context_recall': 0.9667}

In [32]:
results_df = result.to_pandas()
results_df

,question,contexts,answer,ground_truth,context_precision,faithfulness,answer_relevancy,context_recall
0,"```json\n{\n ""question"": ""Considering Michael...","[Patient: 58-year-old male, newly diagnosed Gl...",Whole exome sequencing (WES) plays a crucial r...,Whole exome sequencing plays a crucial role in...,1.0,0.200000,0.845672,0.400000
1,"```json\n{\n ""question"": ""What specific evide...",[Results:\nImmediate Post-Treatment:\n1. Vacci...,I don't know.,The specific evidence from the 6-month follow-...,1.0,0.000000,0.000000,1.000000
2,"```json\n{\n ""question"": ""In the context of t...",[The combination of the neoantigen vaccine wit...,I don't know.,The specific mechanisms that might account for...,1.0,0.000000,0.000000,1.000000
3,"```json\n{\n ""question"": ""What specific modif...",[## Research Summary: Nanotechnology-Based Dru...,I don't know.,The specific modifications made to the nanopar...,1.0,0.000000,0.000000,1.000000
4,"```json\n{\n ""question"": ""Based on the treatm...",[3. In Vivo Studies:\n - Animal model: APP/P...,I don't know.,The provided context does not specify the resu...,0.0,0.333333,0.000000,1.000000
5,"```json\n{\n ""question"": ""Based on the in viv...",[2. In Vivo:\n a) Brain Aβ Levels:\n - ...,I don't know.,The observed reductions in soluble Aβ (67%) an...,1.0,0.000000,0.000000,0.666667
6,"```json\n{\n ""question"": ""What specific prope...",[Discussion:\nThis study demonstrates the pote...,I don't know.,The specific properties of the nanoparticles t...,1.0,0.000000,0.000000,1.000000
7,"```json\n{\n ""question"": ""What specific chall...","[The favorable safety profile is encouraging, ...",I don't know.,The specific challenges related to the animal ...,1.0,0.000000,0.000000,0.500000
8,"```json\n{\n ""question"": ""Considering Sarah T...",[## Case Study 4: CAR-T Cell Therapy for Refra...,CAR-T cell therapy works by modifying a patien...,CAR-T cell therapy involves modifying a patien...,1.0,1.000000,0.812938,1.000000
9,"```json\n{\n ""question"": ""What role does toci...",[Intervention:\nSarah was enrolled in a clinic...,Tocilizumab is utilized in the management of G...,Tocilizumab plays a role in the management of ...,1.0,1.000000,0.820177,1.000000


In [45]:
results_df.head(1)["question"][0]

'```json\n{\n  "question": "Considering Michael\'s involvement in the neoantigen vaccine trial, discuss the significance of whole exome sequencing in identifying potential neoantigens for personalized vaccine development in Glioblastoma Multiforme."\n}\n```'